In [ ]:
BRIDGE_BUILDING_ID = ""

In [1]:
# =============================================================================
# 09_ghg_scope_engine — FABRIC CELL 1 / 2  (DATA PREP)
# Bunu Fabric'te BİR hücreye yapıştır → çalıştır. Sonra CELL 2'yi sonraki hücreye.
# (Tek hücre ~33KB'da kesildiği için 2 parçaya bölündü. Hücreler state paylaşır.)
# Full Scope 1/2/3: refrigerant + market(contract→residual→location) + Cat 1/3/5/6/7/13.
# ÇALIŞMA SIRASI: 03_ref_factors → 04_scope3_demo_activity_generator → BU (cell1+cell2)
# =============================================================================
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, when, coalesce, broadcast,
    date_trunc, year, month,
    sum as spark_sum, round as spark_round, current_timestamp,
)
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, DateType
from delta.tables import DeltaTable
import re

spark.conf.set("spark.sql.shuffle.partitions", "8")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))
spark.conf.set("spark.sql.adaptive.enabled", "true")
print("✅ CELL 1 — Spark konfig tamam")


# --- Lakehouse Tables/ prefix (schema-enabled & flat uyumlu) ---
def _resolve_tables_prefix():
    for t in ["gold_kpi_daily", "silver_building_master", "gold_recommendations"]:
        try:
            sp = spark.table(t).inputFiles()[0]
            m = re.match(r"(abfss://[^/]+@[^/]+/[^/]+)", sp)
            if m:
                base = m.group(1)
                return f"{base}/Tables/dbo" if "/Tables/dbo/" in sp else f"{base}/Tables"
        except Exception:
            continue
    raise Exception("Referans tablo bulunamadı (gold_kpi_daily / silver_building_master). Ana pipeline koştu mu?")


TABLES_PREFIX = _resolve_tables_prefix()
GOLD_KPI_DAILY = f"{TABLES_PREFIX}/gold_kpi_daily"
SILVER_BUILDING = f"{TABLES_PREFIX}/silver_building_master"
OUTPUT_TABLE = f"{TABLES_PREFIX}/gold_ghg_scope"
print(f"✅ Tables prefix: {TABLES_PREFIX[:80]}...")

GAS_BOILER_EFFICIENCY = 0.85   # kazan verimi (model parametresi; faktörler ref katmanından)


# =============================================================================
# BÖLÜM 3 — KAYNAK TABLOLAR + bina master (proxy katmanı)
# =============================================================================
df_kpi = spark.read.format("delta").load(GOLD_KPI_DAILY)
df_building = spark.read.format("delta").load(SILVER_BUILDING)

# --- Bridge scoping (Phase 2): BRIDGE_BUILDING_ID is injected by the orchestrator
#     (param cell above). When set, run for that ONE building; empty = full batch. ---
if BRIDGE_BUILDING_ID:
    df_kpi = df_kpi.filter(f"building_id = '{BRIDGE_BUILDING_ID}'")
    df_building = df_building.filter(f"building_id = '{BRIDGE_BUILDING_ID}'")
    if df_building.count() == 0:
        raise ValueError(f"Bridge: building '{BRIDGE_BUILDING_ID}' not in silver_building_master — run 40_bridge_baseline first.")
    print(f"\U0001f517 BRIDGE MODE: scoped to {BRIDGE_BUILDING_ID}")
_avail = set(df_building.columns)
print(f"✅ kpi {df_kpi.count():,} satır, building {df_building.count()} satır")

if "has_gas_heating" in _avail:
    _gas_expr = col("has_gas_heating").cast("boolean")
else:
    _gas_expr = (coalesce(col("has_hvac_traditional"), lit(False)).cast("boolean")
                 & ~coalesce(col("has_heat_pump"), lit(False)).cast("boolean"))
_diesel_expr = col("has_diesel_generator").cast("boolean") if "has_diesel_generator" in _avail else lit(False)

df_building_slim = df_building.select(
    "building_id",
    col("country_code").alias("country_code"),
    col("conditioned_area_m2").alias("area_m2"),
    _gas_expr.alias("has_gas"),
    _diesel_expr.alias("has_diesel"),
    (col("green_supplier_ef_kg_kwh") if "green_supplier_ef_kg_kwh" in _avail
     else lit(None).cast("double")).alias("supplier_ef"),
    (col("building_type") if "building_type" in _avail else lit("DEFAULT")).alias("building_type"),
    (col("leased_area_m2") if "leased_area_m2" in _avail else lit(None).cast("double")).alias("leased_area_m2"),
).distinct()
print(f"✅ building_slim: {df_building_slim.count()} bina")


# =============================================================================
# BÖLÜM 3b — REFERANS KATMANI + aktivite tabloları (hepsi opsiyonel/graceful)
# =============================================================================
df_ref_grid = (spark.read.format("delta").load(f"{TABLES_PREFIX}/ref_grid_emission_factors")
               .select(col("country_code").alias("ref_country"), col("year").alias("ref_year"),
                       col("emission_factor_kg_kwh").alias("ref_ef")))
_eu = df_ref_grid.filter(col("ref_country") == "EU").orderBy(col("ref_year").desc()).limit(1).collect()
EU_EF_FALLBACK = float(_eu[0]["ref_ef"]) if _eu else 0.23

_fuel = {r["factor_key"]: r["value"]
         for r in spark.read.format("delta").load(f"{TABLES_PREFIX}/ref_fuel_factors").collect()}
GAS_EF_KG_PER_KWH = _fuel.get("natural_gas_kg_per_kwh", 0.201)
DIESEL_EF_KG_PER_LITRE = _fuel.get("diesel_kg_per_litre", 2.68)
# Scope 3 Cat 3 (yakıt&enerji upstream) — WTT + T&D (DEFRA)
ELEC_WTT_KG_KWH = _fuel.get("elec_wtt_kg_per_kwh", 0.0388)
ELEC_TD_KG_KWH = _fuel.get("elec_td_kg_per_kwh", 0.0188)
GAS_WTT_KG_KWH = _fuel.get("gas_wtt_kg_per_kwh", 0.0246)
# Scope 3 Cat 5/6/7 — per-employee faktörler
try:
    _act = {r["factor_key"]: r["value"]
            for r in spark.read.format("delta").load(f"{TABLES_PREFIX}/ref_scope3_activity_factors").collect()}
except Exception:
    _act = {}
COMMUTE_KG_EMP_YR = _act.get("commute_kgco2e_per_emp_yr", 1000.0)
BIZTRAVEL_KG_EMP_YR = _act.get("biztravel_kgco2e_per_emp_yr", 400.0)
WASTE_KG_EMP_YR = _act.get("waste_kgco2e_per_emp_yr", 150.0)
print(f"✅ ref katmanı: grid {df_ref_grid.count()}, gaz {GAS_EF_KG_PER_KWH}, EU fb {EU_EF_FALLBACK}")

# WP2 residual mix (market-based fallback)
try:
    df_ref_residual = (spark.read.format("delta").load(f"{TABLES_PREFIX}/ref_residual_mix")
                       .select(col("country_code").alias("res_country"), col("year").alias("res_year"),
                               col("residual_mix_kg_kwh").alias("res_ef")))
    _has_residual = True
    print(f"✅ ref_residual_mix: {df_ref_residual.count()} satır")
except Exception as e:
    df_ref_residual, _has_residual = None, False
    print(f"ℹ️  ref_residual_mix yok ({type(e).__name__}) → market location'a düşer")

# WP3 refrigerant log → aylık tCO2
try:
    _rl = spark.read.format("delta").load(f"{TABLES_PREFIX}/silver_refrigerant_log")
    _gwp = (spark.read.format("delta").load(f"{TABLES_PREFIX}/ref_refrigerant_gwp")
            .select(col("refrigerant").alias("gwp_refrigerant"), col("gwp_100")))
    df_refrig_monthly = (_rl.withColumn("year_month", date_trunc("month", col("event_date")).cast(DateType()))
                         .join(broadcast(_gwp), col("refrigerant") == col("gwp_refrigerant"), "left")
                         .withColumn("refrig_tco2", spark_round(col("topup_kg") * coalesce(col("gwp_100"), lit(0.0)) / 1000, 4))
                         .groupBy("building_id", "year_month").agg(spark_sum("refrig_tco2").alias("scope1_refrigerant_tco2")))
    _has_refrig = True
    print(f"✅ silver_refrigerant_log: {df_refrig_monthly.count()} bina-ay")
except Exception as e:
    df_refrig_monthly, _has_refrig = None, False
    print(f"ℹ️  silver_refrigerant_log yok ({type(e).__name__}) → refrigerant=0")

# WP4 embodied benchmark (Cat 1)
try:
    df_embodied = (spark.read.format("delta").load(f"{TABLES_PREFIX}/ref_embodied_carbon")
                   .select(col("building_type").alias("emb_type"), col("embodied_kgco2e_m2"), col("amortization_years")))
    _has_embodied = True
    print(f"✅ ref_embodied_carbon: {df_embodied.count()} tip")
except Exception as e:
    df_embodied, _has_embodied = None, False
    print(f"ℹ️  ref_embodied_carbon yok ({type(e).__name__}) → Cat1 default 700")

# WP-max: supplier contracts (market), tenant energy (Cat 13), headcount (Cat 5/6/7)
try:
    df_supplier_c = (spark.read.format("delta").load(f"{TABLES_PREFIX}/silver_supplier_contracts")
                     .select(col("building_id").alias("sc_building"), col("supplier_ef_kg_kwh").alias("contract_supplier_ef")))
    _has_supplier_c = True
    print(f"✅ silver_supplier_contracts: {df_supplier_c.count()} bina")
except Exception as e:
    df_supplier_c, _has_supplier_c = None, False
    print(f"ℹ️  silver_supplier_contracts yok ({type(e).__name__})")

try:
    df_tenant = (spark.read.format("delta").load(f"{TABLES_PREFIX}/silver_tenant_energy")
                 .select("building_id", "year_month", "tenant_kwh"))
    _has_tenant = True
    print(f"✅ silver_tenant_energy: {df_tenant.count()} bina-ay")
except Exception as e:
    df_tenant, _has_tenant = None, False
    print(f"ℹ️  silver_tenant_energy yok ({type(e).__name__}) → Cat13=0")

try:
    df_org = (spark.read.format("delta").load(f"{TABLES_PREFIX}/silver_org_activity")
              .select(col("building_id").alias("org_building"), col("headcount")))
    _has_org = True
    print(f"✅ silver_org_activity: {df_org.count()} bina")
except Exception as e:
    df_org, _has_org = None, False
    print(f"ℹ️  silver_org_activity yok ({type(e).__name__}) → Cat5/6/7=0")


# =============================================================================
# BÖLÜM 4 — AYLIK AGREGASYON
# =============================================================================
df_monthly = (df_kpi.withColumn("year_month", date_trunc("month", col("date")).cast(DateType()))
              .groupBy("building_id", "year_month")
              .agg(spark_sum("total_consumption_kwh").alias("monthly_consumption_kwh"),
                   spark_sum("net_grid_consumption_kwh").alias("monthly_grid_kwh"))
              .withColumn("reporting_year", year(col("year_month")))
              .withColumn("reporting_month", month(col("year_month"))))
print(f"✅ Aylık agregasyon: {df_monthly.count()} satır")


# =============================================================================
# BÖLÜM 5 — JOIN'LER (bina + grid + residual + refrigerant + embodied + supplier + tenant + org)
# =============================================================================
df_joined = df_monthly.join(broadcast(df_building_slim), on="building_id", how="left")

df_with_ef = (df_joined
    .join(broadcast(df_ref_grid),
          (col("country_code") == col("ref_country")) & (col("reporting_year") == col("ref_year")), "left")
    .withColumn("emission_factor_grid", coalesce(col("ref_ef"), lit(EU_EF_FALLBACK)))
    .withColumn("emission_factor_source",
                when(col("ref_ef").isNotNull(), lit("ref_grid_emission_factors")).otherwise(lit("EU_fallback")))
    .drop("ref_country", "ref_year", "ref_ef"))

if _has_residual:
    df_with_ef = (df_with_ef.join(broadcast(df_ref_residual),
        (col("country_code") == col("res_country")) & (col("reporting_year") == col("res_year")), "left")
        .drop("res_country", "res_year"))
else:
    df_with_ef = df_with_ef.withColumn("res_ef", lit(None).cast("double"))

if _has_refrig:
    df_with_ef = df_with_ef.join(df_refrig_monthly, on=["building_id", "year_month"], how="left")
else:
    df_with_ef = df_with_ef.withColumn("scope1_refrigerant_tco2", lit(None).cast("double"))
df_with_ef = df_with_ef.withColumn("scope1_refrigerant_tco2", coalesce(col("scope1_refrigerant_tco2"), lit(0.0)))

if _has_embodied:
    df_with_ef = df_with_ef.join(broadcast(df_embodied), col("building_type") == col("emb_type"), "left").drop("emb_type")
else:
    df_with_ef = (df_with_ef.withColumn("embodied_kgco2e_m2", lit(None).cast("double"))
                  .withColumn("amortization_years", lit(None).cast("int")))

if _has_supplier_c:
    df_with_ef = df_with_ef.join(broadcast(df_supplier_c), col("building_id") == col("sc_building"), "left").drop("sc_building")
else:
    df_with_ef = df_with_ef.withColumn("contract_supplier_ef", lit(None).cast("double"))

if _has_tenant:
    df_with_ef = df_with_ef.join(df_tenant, on=["building_id", "year_month"], how="left")
else:
    df_with_ef = df_with_ef.withColumn("tenant_kwh", lit(None).cast("double"))

if _has_org:
    df_with_ef = df_with_ef.join(broadcast(df_org), col("building_id") == col("org_building"), "left").drop("org_building")
else:
    df_with_ef = df_with_ef.withColumn("headcount", lit(None).cast("int"))

print("✅ CELL 1 tamam — df_with_ef hazır. Şimdi CELL 2'yi sonraki hücreye yapıştır.")


StatementMeta(, f779c2d5-c726-4e9a-81d0-f31d9184474c, 3, Finished, Available, Finished, False)

✅ CELL 1 — Spark konfig tamam
✅ Tables prefix: abfss://6b78345d-d672-40ed-8ac5-258b58d60af9@onelake.dfs.fabric.microsoft.com/8e...
✅ kpi 12,088 satır, building 11 satır
✅ building_slim: 11 bina
✅ ref katmanı: grid 12, gaz 0.201, EU fb 0.242
✅ ref_residual_mix: 8 satır
✅ silver_refrigerant_log: 40 bina-ay
✅ ref_embodied_carbon: 8 tip
✅ silver_supplier_contracts: 11 bina
✅ silver_tenant_energy: 120 bina-ay
✅ silver_org_activity: 11 bina
✅ Aylık agregasyon: 400 satır
✅ CELL 1 tamam — df_with_ef hazır. Şimdi CELL 2'yi sonraki hücreye yapıştır.


In [2]:
# =============================================================================
# 09_ghg_scope_engine — FABRIC CELL 2 / 2  (SCOPE CALC + OUTPUT)
# CELL 1'i çalıştırdıktan SONRA bunu sonraki hücreye yapıştır → çalıştır.
# df_with_ef ve tüm sabitler CELL 1'den gelir (hücreler state paylaşır).
# =============================================================================

# =============================================================================
# BÖLÜM 6 — SCOPE HESAPLAMALARI
# =============================================================================
df_scoped = df_with_ef.withColumn(
    # SCOPE 1 — gaz (ısıtma payı %25 proxy / kazan verimi; gaz faktörü ref'ten)
    "scope1_gas_tco2",
    when(col("has_gas").cast("boolean"),
         spark_round(col("monthly_consumption_kwh") * 0.25 / GAS_BOILER_EFFICIENCY
                     * lit(GAS_EF_KG_PER_KWH) / 1000, 4)).otherwise(lit(0.0))
).withColumn(
    "scope1_diesel_tco2",
    when(col("has_diesel").cast("boolean"),
         spark_round(col("monthly_consumption_kwh") * 0.01 / 10.0
                     * DIESEL_EF_KG_PER_LITRE / 1000, 4)).otherwise(lit(0.0))
).withColumn(
    # WP3: Scope 1 toplam = gaz + dizel + refrigerant fugitive
    "scope1_total_tco2",
    spark_round(col("scope1_gas_tco2") + col("scope1_diesel_tco2") + col("scope1_refrigerant_tco2"), 4)
).withColumn(
    # SCOPE 2 location = net grid × ülke faktörü
    "scope2_location_tco2",
    spark_round(coalesce(col("monthly_grid_kwh"), col("monthly_consumption_kwh"))
                * col("emission_factor_grid") / 1000, 4)
).withColumn(
    # WP2/WP-max: SCOPE 2 market — hiyerarşi: contract → supplier → residual mix → location
    "scope2_market_tco2",
    spark_round(coalesce(col("monthly_grid_kwh"), col("monthly_consumption_kwh"))
                * coalesce(col("contract_supplier_ef"), col("supplier_ef"), col("res_ef"), col("emission_factor_grid"))
                / 1000, 4)
).withColumn(
    "scope2_market_factor",
    coalesce(col("contract_supplier_ef"), col("supplier_ef"), col("res_ef"), col("emission_factor_grid"))
).withColumn(
    "scope2_method",
    when(coalesce(col("contract_supplier_ef"), col("supplier_ef")).isNotNull(), lit("market_based_contract"))
    .when(col("res_ef").isNotNull(), lit("residual_mix_no_instrument"))
    .otherwise(lit("location_fallback_full_disclosure"))
).withColumn(
    # WP4: Scope 3 Cat 1 (embodied) = alan × benchmark / amortizasyon / 12 / 1000
    "scope3_cat1_embodied_tco2",
    spark_round(coalesce(col("area_m2"), lit(0.0)) * coalesce(col("embodied_kgco2e_m2"), lit(700.0))
                / coalesce(col("amortization_years"), lit(60)) / 12.0 / 1000.0, 4)
).withColumn(
    # WP-max: Cat 3 (yakıt&enerji upstream) = grid×(WTT+T&D) + gaz_yakıt×WTT. Ekstra veri YOK.
    "scope3_cat3_fuelenergy_tco2",
    spark_round((coalesce(col("monthly_grid_kwh"), col("monthly_consumption_kwh")) * (lit(ELEC_WTT_KG_KWH) + lit(ELEC_TD_KG_KWH))
                 + when(col("has_gas").cast("boolean"),
                        col("monthly_consumption_kwh") * 0.25 / GAS_BOILER_EFFICIENCY * lit(GAS_WTT_KG_KWH)).otherwise(lit(0.0))
                 ) / 1000.0, 4)
).withColumn(
    # WP-max: Cat 13 (kiracı enerjisi) = tenant_kwh × location faktörü
    "scope3_cat13_leased_tco2",
    spark_round(coalesce(col("tenant_kwh"), lit(0.0)) * col("emission_factor_grid") / 1000.0, 4)
).withColumn(
    # WP-max: Cat 6/7/5 (seyahat/işe gidiş/atık) = headcount × per-emp / 12 / 1000
    "scope3_cat6_travel_tco2",
    spark_round(coalesce(col("headcount"), lit(0)) * lit(BIZTRAVEL_KG_EMP_YR) / 12.0 / 1000.0, 4)
).withColumn(
    "scope3_cat7_commute_tco2",
    spark_round(coalesce(col("headcount"), lit(0)) * lit(COMMUTE_KG_EMP_YR) / 12.0 / 1000.0, 4)
).withColumn(
    "scope3_cat5_waste_tco2",
    spark_round(coalesce(col("headcount"), lit(0)) * lit(WASTE_KG_EMP_YR) / 12.0 / 1000.0, 4)
).withColumn(
    # Scope 3 toplam = Cat 1+3+5+6+7+13 (hâlâ tahmin → disclosure_grade=False)
    "scope3_estimated_tco2",
    spark_round(col("scope3_cat1_embodied_tco2") + col("scope3_cat3_fuelenergy_tco2")
                + col("scope3_cat5_waste_tco2") + col("scope3_cat6_travel_tco2")
                + col("scope3_cat7_commute_tco2") + col("scope3_cat13_leased_tco2"), 4)
).withColumn(
    "total_ghg_location_tco2",
    spark_round(col("scope1_total_tco2") + col("scope2_location_tco2") + col("scope3_estimated_tco2"), 4)
).withColumn(
    "total_ghg_market_tco2",
    spark_round(col("scope1_total_tco2") + col("scope2_market_tco2") + col("scope3_estimated_tco2"), 4)
).withColumn(
    "scope3_method", lit("cat1_embodied+cat3_fuelenergy+cat5_6_7_headcount+cat13_leased")
).withColumn(
    "scope3_disclosure_grade", lit(False)
).withColumn(
    "data_quality_flag",
    when(col("has_gas").cast("boolean") & col("monthly_grid_kwh").isNotNull(), lit("complete"))
    .when(col("has_gas").cast("boolean") & col("monthly_grid_kwh").isNull(), lit("estimated"))
    .otherwise(lit("missing_gas"))
).withColumn("updated_at", current_timestamp())

print("✅ Scope 1 / 2 / 3 (6 kategori) hesaplandı")


# =============================================================================
# BÖLÜM 7 — ÇIKTI TABLOSU
# =============================================================================
FINAL_COLS = [
    "building_id", "year_month", "reporting_year", "reporting_month",
    "scope1_gas_tco2", "scope1_diesel_tco2", "scope1_refrigerant_tco2", "scope1_total_tco2",
    "scope2_location_tco2", "scope2_market_tco2", "scope2_market_factor", "scope2_method",
    "scope3_cat1_embodied_tco2", "scope3_cat3_fuelenergy_tco2", "scope3_cat5_waste_tco2",
    "scope3_cat6_travel_tco2", "scope3_cat7_commute_tco2", "scope3_cat13_leased_tco2",
    "scope3_estimated_tco2", "scope3_method", "scope3_disclosure_grade",
    "total_ghg_location_tco2", "total_ghg_market_tco2",
    "emission_factor_grid", "emission_factor_source", "data_quality_flag", "updated_at",
]
df_final = df_scoped.select(FINAL_COLS)

print("\n📊 Çıktı özeti (portföy toplamı):")
df_final.agg(
    spark_sum("scope1_total_tco2").alias("Scope1"),
    spark_sum("scope2_location_tco2").alias("Scope2_Location"),
    spark_sum("scope2_market_tco2").alias("Scope2_Market"),
    spark_sum("scope3_estimated_tco2").alias("Scope3"),
    spark_sum("total_ghg_location_tco2").alias("Total_Location"),
).show()


# =============================================================================
# BÖLÜM 8 — DELTA MERGE (idempotent), tablo yoksa overwrite fallback
# =============================================================================
_tbl = OUTPUT_TABLE.rstrip("/").split("/")[-1]
try:
    gold_table = DeltaTable.forPath(spark, OUTPUT_TABLE)
    (gold_table.alias("tgt").merge(
        df_final.alias("src"),
        "tgt.building_id = src.building_id AND tgt.year_month = src.year_month")
     .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
    print(f"✅ Delta MERGE tamam: {_tbl}")
except Exception as e:
    print(f"ℹ️  MERGE atlandı ({type(e).__name__}: {str(e)[:120]}) → ilk kez overwrite")
    (df_final.write.format("delta").partitionBy("reporting_year")
     .mode("overwrite").option("overwriteSchema", "true").saveAsTable(_tbl))
    print(f"✅ Tablo oluşturuldu + kataloga kaydedildi: {_tbl}")


# =============================================================================
# BÖLÜM 9 — OPTIMIZE + katalog sync
# =============================================================================
try:
    spark.sql(f"OPTIMIZE delta.`{OUTPUT_TABLE}` ZORDER BY (building_id, year_month)")
    print("✅ Z-ORDER OPTIMIZE tamam")
except Exception as e:
    print(f"⚠️  OPTIMIZE atlandı (yeni tabloda normal): {type(e).__name__}")

try:
    spark.catalog.refreshTable(_tbl)
    print(f"✅ Katalog cache temizlendi: {_tbl}")
except Exception:
    print(f"ℹ️  '{_tbl}' katalogda 1-2 dk içinde keşfedilir")

df_check = spark.read.format("delta").load(OUTPUT_TABLE)
print(f"\n📊 gold_ghg_scope: {df_check.count():,} satır, {df_check.select('building_id').distinct().count()} bina")
print("✅ CELL 2 tamam — gold_ghg_scope güncel (full Scope 1/2/3, 6 kategori).")


StatementMeta(, f779c2d5-c726-4e9a-81d0-f31d9184474c, 4, Finished, Available, Finished, False)

✅ Scope 1 / 2 / 3 (6 kategori) hesaplandı

📊 Çıktı özeti (portföy toplamı):
+-----------------+---------------+-----------------+-----------------+-----------------+
|           Scope1|Scope2_Location|    Scope2_Market|           Scope3|   Total_Location|
+-----------------+---------------+-----------------+-----------------+-----------------+
|2981.390399999999|     35545.2446|56119.35019999996|29833.09779999999|68359.73280000006|
+-----------------+---------------+-----------------+-----------------+-----------------+

✅ Delta MERGE tamam: gold_ghg_scope
✅ Z-ORDER OPTIMIZE tamam
✅ Katalog cache temizlendi: gold_ghg_scope

📊 gold_ghg_scope: 400 satır, 10 bina
✅ CELL 2 tamam — gold_ghg_scope güncel (full Scope 1/2/3, 6 kategori).


In [3]:
# =============================================================================
# 09_ghg — CELL 3 / 3  (BREAKDOWN LONG — scope×category boyutu, drill için)
# cell1 + cell2'den SONRA çalıştır (ayrı bir hücreye yapıştır).
# gold_ghg_scope'u "uzun format"a çevirir → category GERÇEK bir kolon olur
# (DirectLake'te calc-column gerekmez). Tek ölçü + drill + temiz görsel sağlar.
# =============================================================================
from pyspark.sql.functions import col, lit
from functools import reduce
import re


def _prefix():
    for t in ["gold_ghg_scope", "gold_kpi_daily", "silver_building_master"]:
        try:
            sp = spark.table(t).inputFiles()[0]
            m = re.match(r"(abfss://[^/]+@[^/]+/[^/]+)", sp)
            if m:
                base = m.group(1)
                return f"{base}/Tables/dbo" if "/Tables/dbo/" in sp else f"{base}/Tables"
        except Exception:
            continue
    raise Exception("gold_ghg_scope bulunamadı — önce cell1 + cell2 çalıştır.")


P = _prefix()
df = spark.read.format("delta").load(f"{P}/gold_ghg_scope")

# (scope, category, kolon, data_grade) — additive bileşenler (Scope 2 = location headline)
pairs = [
    ("Scope 1", "Gas",                    "scope1_gas_tco2",            "core"),
    ("Scope 1", "Diesel",                 "scope1_diesel_tco2",         "core"),
    ("Scope 1", "Refrigerant",            "scope1_refrigerant_tco2",    "estimated"),
    ("Scope 2", "Electricity (location)", "scope2_location_tco2",       "core"),
    ("Scope 3", "Cat 1 Embodied",         "scope3_cat1_embodied_tco2",  "estimated"),
    ("Scope 3", "Cat 3 Fuel & Energy",    "scope3_cat3_fuelenergy_tco2","estimated"),
    ("Scope 3", "Cat 5 Waste",            "scope3_cat5_waste_tco2",     "estimated"),
    ("Scope 3", "Cat 6 Travel",           "scope3_cat6_travel_tco2",    "estimated"),
    ("Scope 3", "Cat 7 Commute",          "scope3_cat7_commute_tco2",   "estimated"),
    ("Scope 3", "Cat 13 Leased",          "scope3_cat13_leased_tco2",   "estimated"),
]
parts = [
    df.select(
        "building_id", "year_month", "reporting_year", "reporting_month",
        lit(s).alias("scope"), lit(c).alias("category"),
        lit(g).alias("data_grade"), col(cn).alias("value_tco2"),
    )
    for s, c, cn, g in pairs
]
df_long = reduce(lambda a, b: a.unionByName(b), parts)

_tbl = "gold_ghg_breakdown_long"
df_long.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(_tbl)
try:
    spark.catalog.refreshTable(_tbl)
except Exception:
    pass

print(f"✅ {_tbl}: {df_long.count()} satır (scope × category long format).")
print("   Modele ekle → tek ölçü [Emissions tCO2e]=SUM(value_tco2) → drill/slicer ile temiz görsel.")
df_long.groupBy("scope", "category").count().orderBy("scope", "category").show(20, truncate=False)


StatementMeta(, f779c2d5-c726-4e9a-81d0-f31d9184474c, 5, Finished, Available, Finished, False)

✅ gold_ghg_breakdown_long: 4000 satır (scope × category long format).
   Modele ekle → tek ölçü [Emissions tCO2e]=SUM(value_tco2) → drill/slicer ile temiz görsel.
+-------+----------------------+-----+
|scope  |category              |count|
+-------+----------------------+-----+
|Scope 1|Diesel                |400  |
|Scope 1|Gas                   |400  |
|Scope 1|Refrigerant           |400  |
|Scope 2|Electricity (location)|400  |
|Scope 3|Cat 1 Embodied        |400  |
|Scope 3|Cat 13 Leased         |400  |
|Scope 3|Cat 3 Fuel & Energy   |400  |
|Scope 3|Cat 5 Waste           |400  |
|Scope 3|Cat 6 Travel          |400  |
|Scope 3|Cat 7 Commute         |400  |
+-------+----------------------+-----+

